In [28]:
import requests
import json
import pandas as pd

url = "https://us30.vintrace.net/bla/api/v6/inventory"
headers = {
    "Accept": "application/json",
    "Authorization": "Token",
    "correlation-id": ""
}

# Adding query parameters for filtering by owner to reduce initial data pull
params = {
    "filter[owner]": "Blackbird Vineyards"
}

# Fetching inventory summaries with owner filter
response = requests.get(url, headers=headers, params=params)

if response.status_code == 200:
    data = response.json()
    inventory_summaries = data.get("inventorySummaries", [])

    filtered_items = []
    if inventory_summaries:
        for item in inventory_summaries:
            bond = item.get("bond", "")
            # Further filter for items where 'bond' contains 'Blackbird Vineyards'
            if "Blackbird Vineyards" in bond:
                filtered_items.append(item)

    if filtered_items:
        print(f"Found {len(filtered_items)} items with 'Blackbird Vineyards' in their bond.")

        # Prepare data for DataFrame
        df_data = []
        for item in filtered_items:
            df_data.append({
                "Name": item.get("name", "N/A"),
                "Vintage": item.get("vintage", "N/A"),
                "Quantity": item.get("quantity", "N/A"),
                "Total Cost (Text)": item.get("totalCostAsText", "N/A")
            })

        # Create DataFrame
        df = pd.DataFrame(df_data)
        print("\nDataFrame created with selected fields:")
        display(df.head())

    else:
        print(f"No items found with 'Blackbird Vineyards' in their bond after filtering by owner.")
else:
    print(f"Error {response.status_code}: {response.text}")

Found 85 items with 'Blackbird Vineyards' in their bond.

DataFrame created with selected fields:


,Name,Vintage,Quantity,Total Cost (Text)
0,BBCF25027HAY/BLK,2025,360.0,"$27,471.474"
1,BBCF25035HUD1/BLK,2025,840.0,"$51,155.8295"
2,BBCF25036.HUD2/BLK,2025,780.0,"$22,942.3951"
3,BBCF25037HUD3/BLK,2025,780.0,$137.7347
4,BBCF25038LARK/BLK,2025,720.0,"$59,274.7517"


In [61]:
import requests
import json
import pandas as pd

url = "https://us30.vintrace.net/bla/api/v6/inventory"
headers = {
    "Accept": "application/json",
    "Authorization": "Token",
    "correlation-id": ""
}

# Adding query parameters for filtering by owner to reduce initial data pull
params = {
    "filter[owner]": "Blackbird Vineyards"
}

# Fetching inventory summaries with owner filter
response = requests.get(url, headers=headers, params=params)

if response.status_code == 200:
    data = response.json()
    inventory_summaries = data.get("inventorySummaries", [])

    filtered_items = []
    if inventory_summaries:
        for item in inventory_summaries:
            bond = item.get("bond", "")
            # Further filter for items where 'bond' contains 'Blackbird Vineyards'
            if "Blackbird Vineyards" in bond:
                filtered_items.append(item)

    if filtered_items:
        print(f"Found {len(filtered_items)} items with 'Blackbird Vineyards' in their bond.")

        # Prepare data for DataFrame
        df_data = []
        for item in filtered_items:
            df_data.append({
                "Name": item.get("name", "N/A"),
                "Vintage": item.get("vintage", "N/A"),
                "Quantity": item.get("quantity", "N/A"),
                "Total Cost (Text)": item.get("totalCostAsText", "N/A")
            })

        # Create DataFrame
        df = pd.DataFrame(df_data)
        print("\nDataFrame created with selected fields:")
        display(df.head())

    else:
        print(f"No items found with 'Blackbird Vineyards' in their bond after filtering by owner.")
else:
    print(f"Error {response.status_code}: {response.text}")

Found 84 items with 'Blackbird Vineyards' in their bond.

DataFrame created with selected fields:


,Name,Vintage,Quantity,Total Cost (Text)
0,BBCF25027HAY/BLK,2025,360.0,"$27,471.474"
1,BBCF25035HUD1/BLK,2025,840.0,"$51,155.8295"
2,BBCF25036.HUD2/BLK,2025,780.0,"$22,942.3951"
3,BBCF25037HUD3/BLK,2025,780.0,$137.7347
4,BBCF25038LARK/BLK,2025,720.0,"$59,274.7517"


#Inventory Items

There should be the two columns under inventory.

Then we should make a finished wines category. These all end in 75.


In [36]:
import re

# Define a function to check for the finished wine pattern
def is_finished_wine(name):
    if isinstance(name, str):
        # New pattern: checks if '75' or '75/BLK' is present at the end of the name
        return bool(re.search(r'(75|75/BLK)$', name))
    return False

# Apply the function to create the new column
df['Is Finished Wine'] = df['Name'].apply(is_finished_wine)

# Filter the DataFrame to show only finished wines and explicitly create a copy
finished_wines_df = df[df['Is Finished Wine'] == True].copy()

print(f"Found {len(finished_wines_df)} finished wines.")
display(finished_wines_df.head())

Found 10 finished wines.


,Name,Vintage,Quantity,Total Cost (Text),Is Finished Wine
36,BBVA2375,None,3542.0,"$638,360.5755",True
38,BBVAR2575/BLK,2025,1089.0,"$6,519.3845",True
39,BBVAR2575/BLK,2025,1931.0,"$11,560.084",True
40,BBVAR2575/BLK,2025,1931.0,"$11,560.084",True
51,BBVCH2475/BLK,2024,33.7,$886.5991,True


In [37]:
import numpy as np

# Convert 'Total Cost (Text)' to a numeric column
# Remove '$' and ',' then convert to float
finished_wines_df['Total Cost Numeric'] = finished_wines_df['Total Cost (Text)'].replace('[$,]', '', regex=True).astype(float)

# Define conversion factors
gallons_per_case = 2.4
bottles_per_case = 12

# Calculate the number of cases and bottles
# Handle potential division by zero for quantity
finished_wines_df['Cases'] = np.where(
    finished_wines_df['Quantity'] > 0,
    finished_wines_df['Quantity'] / gallons_per_case,
    0
)
finished_wines_df['Bottles'] = finished_wines_df['Cases'] * bottles_per_case

# Calculate Cost per Case and Cost per Bottle
# Handle potential division by zero for cases/bottles
finished_wines_df['Cost per Case'] = np.where(
    finished_wines_df['Cases'] > 0,
    finished_wines_df['Total Cost Numeric'] / finished_wines_df['Cases'],
    0
)
finished_wines_df['Cost per Bottle'] = np.where(
    finished_wines_df['Bottles'] > 0,
    finished_wines_df['Total Cost Numeric'] / finished_wines_df['Bottles'],
    0
)

# Display the updated DataFrame with the new cost columns
print("DataFrame with Cost per Case and Cost per Bottle:")
display(finished_wines_df.head())

DataFrame with Cost per Case and Cost per Bottle:


,Name,Vintage,Quantity,Total Cost (Text),Is Finished Wine,Total Cost Numeric,Cases,Bottles,Cost per Case,Cost per Bottle
36,BBVA2375,None,3542.0,"$638,360.5755",True,638360.5755,1475.833333,17710.0,432.542457,36.045205
38,BBVAR2575/BLK,2025,1089.0,"$6,519.3845",True,6519.3845,453.750000,5445.0,14.367790,1.197316
39,BBVAR2575/BLK,2025,1931.0,"$11,560.084",True,11560.0840,804.583333,9655.0,14.367790,1.197316
40,BBVAR2575/BLK,2025,1931.0,"$11,560.084",True,11560.0840,804.583333,9655.0,14.367790,1.197316
51,BBVCH2475/BLK,2024,33.7,$886.5991,True,886.5991,14.041667,168.5,63.140589,5.261716


In [62]:
display(finished_wines_df)

,Name,Vintage,Quantity,Total Cost (Text),Is Finished Wine,Total Cost Numeric,Cases,Bottles,Cost per Case,Cost per Bottle
36,BBVA2375,None,3542.0,"$638,360.5755",True,638360.5755,1475.833333,17710.0,432.542457,36.045205
38,BBVAR2575/BLK,2025,1089.0,"$6,519.3845",True,6519.3845,453.750000,5445.0,14.367790,1.197316
39,BBVAR2575/BLK,2025,1931.0,"$11,560.084",True,11560.0840,804.583333,9655.0,14.367790,1.197316
40,BBVAR2575/BLK,2025,1931.0,"$11,560.084",True,11560.0840,804.583333,9655.0,14.367790,1.197316
51,BBVCH2475/BLK,2024,33.7,$886.5991,True,886.5991,14.041667,168.5,63.140589,5.261716
52,BBVCH2475/BLK,2024,956.0,"$25,151.0006",True,25151.0006,398.333333,4780.0,63.140587,5.261716
63,BBVI2375,None,100.0,"$16,702.0564",True,16702.0564,41.666667,500.0,400.849354,33.404113
76,BBVP2375,None,100.0,"$16,236.5985",True,16236.5985,41.666667,500.0,389.678364,32.473197
79,BBVSB2575/BLK,2025,2062.0,"$18,000.0001",True,18000.0001,859.166667,10310.0,20.950534,1.745878
83,BBVSBP2575/BLK,2025,629.5,"$17,792.5638",True,17792.5638,262.291667,3147.5,67.835033,5.652919


In [63]:
import http.client

conn = http.client.HTTPSConnection("us30.vintrace.net")

headers = {
    'correlation-id': "",
    'Accept': "application/json",
    'Authorization': "Bearer Token"
}

conn.request("GET", "/bla/api/v7/operation/wine-batches", headers=headers)

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))

{"totalResults":1153,"offset":0,"limit":10,"first":"/bla/api/v7/operation/wine-batches?limit=10&offset=0","previous":null,"next":"/bla/api/v7/operation/wine-batches?limit=10&offset=10","last":"/bla/api/v7/operation/wine-batches?limit=10&offset=1150","results":[{"id":1,"batchCode":"Many to Many Blend Batch","batchNumber":null,"description":null,"productionYear":2017,"owner":{"extId":null,"id":3,"name":"Blackbird Vineyards"},"grading":null,"program":null,"designatedRegion":null,"designatedSubRegion":null,"designatedVariety":null,"winery":{"businessUnit":null,"id":1,"name":"Winery"},"category":null,"designatedProduct":null,"costsTrackedPercentage":100.0,"ageOfSpirits":null,"serviceOrder":null,"fractionType":null,"inactive":true,"vessels":[],"allocations":[]},{"id":2,"batchCode":"BKBCBF7466PRES","batchNumber":null,"description":"2017 Krupp CBF press","productionYear":2017,"owner":{"extId":null,"id":3,"name":"Blackbird Vineyards"},"grading":null,"program":null,"designatedRegion":{"id":29,"n

In [66]:
import json

# Assuming 'data' from the previous cell holds the raw byte response
# Decode the bytes to a string, then parse as JSON
wine_batch_data = json.loads(data.decode("utf-8"))

wine_batches = wine_batch_data.get("results", [])

inactive_batches = []
if wine_batches:
    for batch in wine_batches:
        if batch.get("inactive") is True:
            inactive_batches.append(batch)

if inactive_batches:
    print(f"Found {len(inactive_batches)} inactive wine batches:")
    for batch in inactive_batches:
        owner = batch.get("owner", {})
        owner_name = owner.get("name", "N/A")
        print(f"  Batch ID: {batch.get('id')}, Code: {batch.get('batchCode')}, Year: {batch.get('productionYear')}, Owner: {owner_name}")
else:
    print("No inactive wine batches found in the response.")

Found 1 inactive wine batches:
  Batch ID: 1, Code: Many to Many Blend Batch, Year: 2017, Owner: Blackbird Vineyards


In [67]:
import requests
import json

# Base URL for the wine batches API
base_url = "https://us30.vintrace.net/bla/api/v7/operation/wine-batches"

# Headers for API authentication
headers = {
    "Accept": "application/json",
    "Authorization": "Bearer Token",
    "correlation-id": ""
}

all_wine_batches = []
limit = 100  # Set a reasonable limit for each request
offset = 0
total_results = float('inf') # Initialize with a large number to enter the loop

print("Fetching all wine batches...")

while offset < total_results:
    params = {
        "limit": limit,
        "offset": offset
    }

    try:
        response = requests.get(base_url, headers=headers, params=params)
        response.raise_for_status()  # Raise an exception for HTTP errors
        data = response.json()

        # Update total_results from the first response
        if offset == 0:
            total_results = data.get("totalResults", 0)
            print(f"Total wine batches to fetch: {total_results}")

        current_batches = data.get("results", [])
        all_wine_batches.extend(current_batches)

        offset += limit
        print(f"Fetched {len(all_wine_batches)}/{total_results} batches...")

    except requests.exceptions.HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
        print(f"Response content: {response.text}")
        break
    except requests.exceptions.ConnectionError as conn_err:
        print(f"Connection error occurred: {conn_err}")
        break
    except requests.exceptions.Timeout as timeout_err:
        print(f"Timeout error occurred: {timeout_err}")
        break
    except requests.exceptions.RequestException as req_err:
        print(f"An unexpected error occurred: {req_err}")
        break
    except json.JSONDecodeError:
        print(f"Failed to decode JSON from response: {response.text}")
        break

print(f"Finished fetching. Total wine batches collected: {len(all_wine_batches)}")


Fetching all wine batches...
Total wine batches to fetch: 1153
Fetched 100/1153 batches...
Fetched 200/1153 batches...
Fetched 300/1153 batches...
Fetched 400/1153 batches...
Fetched 500/1153 batches...
Fetched 600/1153 batches...
Fetched 700/1153 batches...
Fetched 800/1153 batches...
Fetched 900/1153 batches...
Fetched 1000/1153 batches...
Fetched 1100/1153 batches...
Fetched 1153/1153 batches...
Finished fetching. Total wine batches collected: 1153


In [69]:
blackbird_vineyards_batches_all = []

if all_wine_batches:
    for batch in all_wine_batches:
        owner = batch.get("owner", {})
        owner_name = owner.get("name")
        if owner_name == "Blackbird Vineyards":
            blackbird_vineyards_batches_all.append(batch)

if blackbird_vineyards_batches_all:
    print(f"Found {len(blackbird_vineyards_batches_all)} wine batches owned by Blackbird Vineyards (from all paginated results):")
    # Display a sample of these batches
    for batch in blackbird_vineyards_batches_all[:5]: # Display first 5 as a sample
        print(f"  Batch ID: {batch.get('id')}, Code: {batch.get('batchCode')}, Year: {batch.get('productionYear')}")
else:
    print("No wine batches found owned by 'Blackbird Vineyards' across all paginated results.")

Found 472 wine batches owned by Blackbird Vineyards (from all paginated results):
  Batch ID: 1, Code: Many to Many Blend Batch, Year: 2017
  Batch ID: 2, Code: BKBCBF7466PRES, Year: 2017
  Batch ID: 180, Code: BBVMER8007BARK, Year: 2018
  Batch ID: 183, Code: BBVMER8009STAD4, Year: 2018
  Batch ID: 186, Code: BBVMER8008BROK, Year: 2018


In [70]:
import re

finished_blackbird_batches = []

if blackbird_vineyards_batches_all:
    for batch in blackbird_vineyards_batches_all:
        batch_code = batch.get("batchCode", "")
        # Use the regex pattern to check if batchCode ends with '75' or '75/BLK'
        if re.search(r'(75|75/BLK)$', batch_code):
            finished_blackbird_batches.append(batch)

if finished_blackbird_batches:
    print(f"Found {len(finished_blackbird_batches)} Blackbird Vineyards batches ending in '75' or '75/BLK':")
    # Display a sample of these batches
    for batch in finished_blackbird_batches[:5]: # Display first 5 as a sample
        owner = batch.get("owner", {})
        owner_name = owner.get("name", "N/A")
        print(f"  Batch ID: {batch.get('id')}, Code: {batch.get('batchCode')}, Year: {batch.get('productionYear')}, Owner: {owner_name}")
else:
    print("No Blackbird Vineyards batches found ending in '75' or '75/BLK'.")

Found 20 Blackbird Vineyards batches ending in '75' or '75/BLK':
  Batch ID: 17995, Code: BBVI2175, Year: 2021, Owner: Blackbird Vineyards
  Batch ID: 17998, Code: BBVC2175, Year: 2021, Owner: Blackbird Vineyards
  Batch ID: 18000, Code: BBVP2175, Year: 2021, Owner: Blackbird Vineyards
  Batch ID: 22428, Code: BBVI2275, Year: 2022, Owner: Blackbird Vineyards
  Batch ID: 22464, Code: BBVA2275, Year: 2022, Owner: Blackbird Vineyards
